# Team: Conniving Kangaroo
## System: ClueLLM

Story generation system using Template 1: Suspense Generation

### Technical Details:
- Utilizes Google's Gemini 2.5 Flash
- Please use your own API key (see README)
- Typical run time for all cells in this notebook: ~5-10 minutes
- Typical cost for API calls: ~15 cents

In [ ]:
!pip install -q -U google-genai

## LLM Setup

Define Gemini model API and LLM helper methods.

In [ ]:
import os
# replace with your own API key
os.environ["GOOGLE_API_KEY"] = ""

In [ ]:
from google import genai

client = genai.Client()
MODEL = "gemini-2.5-flash"

# Sanity check
response = client.models.generate_content(model=MODEL, contents="Say: ready")
print("LLM status:", response.text.strip())

LLM status: ready


In [ ]:
def ask_llm(prompt: str):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return response.text.strip()

In [ ]:
def extract_json(text: str):
    """
    Extract the first JSON object or array from model output.
    Handles markdown code fences like ```json ... ```.
    """
    # Remove markdown fences if present
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    # Try full parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: find first JSON object or array
    match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))

    raise ValueError(f"No JSON found in LLM response:\n{text}")

## Data Structures
Defines all dataclasses used to store story state. Sections 2 and 3 (story generation and compilation) will populate these.

| Class | Role | Function |
|---|---|---|
| `CrimeDetails` | The hidden truth | Keeps the ground-truth crime facts isolated so the LLM can never accidentally leak them into suspect dialogue or clue descriptions |
| `Character` | Any person in the story | Stores means/motive/opportunity explicitly so the meta-controller can check MMO coverage without re-querying the LLM |
| `Clue` | A piece of evidence | `is_red_herring` and `discovered` flags let the controller track what the detective *can* know vs. what *really* points to the criminal |
| `PlotPoint` | One story beat | `suspense_score` on each beat makes the rising tension arc a measurable, enforceable property rather than a vague stylistic goal |
| `StoryWorld` | Global state container | A single object passed by reference into every generation function — any LLM call can read current state via `summary_so_far()` without duplicating data |

In [ ]:
import json
import re
from dataclasses import dataclass, field
from typing import List, Optional
from pprint import pprint


@dataclass
class Character:
    """A character in the story: detective, suspect, criminal, or witness."""
    name: str
    role: str                        # 'detective' | 'suspect' | 'criminal' | 'witness'
    description: str
    means: str                       # capability to commit the crime
    motive: str                      # reason to commit the crime
    opportunity: str                 # could they have been at the scene?
    alibi: str
    alibi_is_lie: bool = False
    is_criminal: bool = False
    eliminated: bool = False         # True once the detective rules them out
    notes: List[str] = field(default_factory=list)


@dataclass
class Clue:
    """A piece of evidence or information discoverable by the detective."""
    clue_id: int
    description: str
    location: str
    discovered: bool = False
    is_red_herring: bool = False
    points_to: str = ""              # character or event this implicates
    revealed_in_plot_point: int = -1


@dataclass
class PlotPoint:
    """One story beat that advances the plot."""
    index: int
    title: str
    description: str
    action_taken: str
    outcome: str
    obstacle: str
    suspense_score: float            # 0.0 (low tension) → 1.0 (maximum)
    characters_involved: List[str] = field(default_factory=list)
    clues_revealed: List[int] = field(default_factory=list)
    clues_used: List[int] = field(default_factory=list)


@dataclass
class CrimeDetails:
    """The hidden backstory: what really happened before the detective arrives."""
    crime_type: str
    victim: str
    location: str
    time_of_crime: str
    method: str
    criminal_name: str
    criminal_motive: str
    criminal_means: str
    key_evidence_hidden: List[str] = field(default_factory=list)
    backstory_summary: str = ""


@dataclass
class StoryWorld:
    """Master container for all story state."""
    crime_details: Optional[CrimeDetails] = None
    characters: List[Character] = field(default_factory=list)
    clues: List[Clue] = field(default_factory=list)
    plot_points: List[PlotPoint] = field(default_factory=list)
    countdown_deadline: str = ""
    countdown_consequence: str = ""
    final_narrative: str = ""

    def get_character(self, name: str) -> Optional[Character]:
        for c in self.characters:
            if c.name.lower() == name.lower():
                return c
        return None

    def get_clue(self, clue_id: int) -> Optional[Clue]:
        for cl in self.clues:
            if cl.clue_id == clue_id:
                return cl
        return None

    def discovered_clues(self) -> List[Clue]:
        return [cl for cl in self.clues if cl.discovered]

    def active_suspects(self) -> List[Character]:
        return [c for c in self.characters if c.role in ('suspect', 'criminal') and not c.eliminated]

    def summary_so_far(self) -> str:
        """Brief text digest of current story state, used as context in LLM prompts."""
        lines = []
        if self.crime_details:
            cd = self.crime_details
            lines.append(f"CRIME: {cd.crime_type} of {cd.victim} at {cd.location} ({cd.time_of_crime}).")
        lines.append(f"ACTIVE SUSPECTS: {[c.name for c in self.active_suspects()]}")
        lines.append(f"CLUES FOUND: {[cl.description[:40] for cl in self.discovered_clues()]}")
        if self.plot_points:
            last = self.plot_points[-1]
            lines.append(f"LAST PLOT POINT ({last.index}): {last.title} — {last.outcome}")
        return "\n".join(lines)


print("Data structures defined.")


Data structures defined.


In [ ]:
# Initialize StoryWorld object
world = StoryWorld()
print("StoryWorld initialized:", world)


StoryWorld initialized: StoryWorld(crime_details=None, characters=[], clues=[], plot_points=[], countdown_deadline='', countdown_consequence='', final_narrative='')


## Phase 1: Crime Backstory

In [ ]:
# Builds the prompt for the LLM to generate the crime details
def build_crime_details_prompt() -> str:
    return """
You are helping build the hidden ground-truth backstory for a gala murder crime mystery.

Generate a single crime scenario for a detective story.

Requirements:
- The crime must be a serious crime suitable for a mystery story.
- It must be solvable, but not obvious.
- The crime must support multiple plausible suspects later.
- The criminal's motive, means, and opportunity must all make sense together.
- The method of the crime must leave behind evidence that can later be discovered.
- The crime should not rely on supernatural elements or impossible coincidences.

Return ONLY valid JSON with exactly these keys:
{
  "crime_type": "...",
  "victim": "...",
  "location": "...",
  "time_of_crime": "...",
  "method": "...",
  "criminal_name": "...",
  "criminal_motive": "...",
  "criminal_means": "...",
  "key_evidence_hidden": ["...", "...", "..."],
  "backstory_summary": "..."
}

Field guidance:
- crime_type: e.g. "murder", "blackmail", "kidnapping", "art theft" (in this case, it will be "murder")
- victim: full name and a few identifying words
- location: specific and story-friendly
- time_of_crime: specific enough to anchor a timeline
- method: how the crime was carried out
- criminal_name: full name
- criminal_motive: concrete and personal
- criminal_means: how they were capable of doing it
- key_evidence_hidden: 3 to 5 important hidden evidence items
- backstory_summary: 1 paragraph summary of what truly happened before the detective starts investigating

Important:
- The criminal should not be the most obvious suspect.
- The motive should be meaningful, not generic.
- The evidence should connect naturally to the crime method.
- Return JSON only. No explanation.
""".strip()


# Prompts the LLM and extracts the answer to a data structure
def generate_crime_details() -> CrimeDetails:
    prompt = build_crime_details_prompt()
    raw = ask_llm(prompt)
    data = extract_json(raw)

    crime = CrimeDetails(
        crime_type=data["crime_type"],
        victim=data["victim"],
        location=data["location"],
        time_of_crime=data["time_of_crime"],
        method=data["method"],
        criminal_name=data["criminal_name"],
        criminal_motive=data["criminal_motive"],
        criminal_means=data["criminal_means"],
        key_evidence_hidden=data.get("key_evidence_hidden", []),
        backstory_summary=data.get("backstory_summary", "")
    )
    return crime

In [ ]:
# Run crime detail generation
world.crime_details = generate_crime_details()

pprint(world.crime_details)
print()
print(world.crime_details.backstory_summary)

CrimeDetails(crime_type='murder',
             victim='Alistair Finch, 62, a renowned, cutthroat art collector '
                    'and philanthropist',
             location="Alistair Finch's private lounge, adjacent to the Grand "
                      'Ballroom of the Azure Bloom Hotel',
             time_of_crime="10:45 PM, during the annual 'Gala for the Arts'",
             method='Poisoning by inhalation. The criminal injected a highly '
                    'concentrated, fast-acting respiratory depressant into the '
                    "victim's custom vape pen liquid, knowing he would use it "
                    'shortly after his speech.',
             criminal_name='Eleanor Vance',
             criminal_motive="Alistair Finch had uncovered Eleanor's long-term "
                             "embezzlement of funds from his 'Finch Arts "
                             "Endowment' charity. He planned to publicly "
                             'expose her at the gala, which woul

## Phase 2: Characters & Clues

In [ ]:
# Builds the prompt for generating character list
def build_characters_prompt(crime: CrimeDetails) -> str:
    crime_payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "criminal_means": crime.criminal_means,
        "key_evidence_hidden": crime.key_evidence_hidden,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are generating the cast of characters for a crime mystery.

The hidden ground-truth crime is:
{json.dumps(crime_payload, indent=2)}

Create:
- 1 detective
- 5 suspects who are NOT the criminal
- 1 criminal character whose name MUST exactly match the criminal_name above

Return ONLY valid JSON with exactly this structure:
{{
  "detective": {{
    "name": "...",
    "role": "detective",
    "description": "...",
    "means": "N/A",
    "motive": "solve the case",
    "opportunity": "arrives after the crime to investigate",
    "alibi": "N/A"
  }},
  "criminal": {{
    "name": "...",
    "role": "criminal",
    "description": "...",
    "means": "...",
    "motive": "...",
    "opportunity": "...",
    "alibi": "...",
    "alibi_is_lie": true
  }},
  "suspects": [
    {{
      "name": "...",
      "role": "suspect",
      "description": "...",
      "means": "...",
      "motive": "...",
      "opportunity": "...",
      "alibi": "...",
      "alibi_is_lie": false
    }}
  ]
}}

Requirements:
- The criminal name must exactly match the given criminal_name.
- There must be exactly 5 suspects in the suspects list.
- Every suspect must have plausible means, motive, and opportunity.
- At least 2 suspects should look strongly suspicious at first.
- At least 2 innocent suspects should have a false or misleading alibi.
- The criminal should not be the most obvious suspect.
- The detective should feel capable and specific, not generic.
- Suspects should have relationships to the victim, location, or circumstances of the crime.
- Keep all details realistic and internally consistent.

Return JSON only.
""".strip()


# Prompts the LLM and extracts the character data to our data structure
def generate_characters(world: StoryWorld) -> List[Character]:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist before generating characters.")

    raw = ask_llm(build_characters_prompt(world.crime_details))
    data = extract_json(raw)

    characters = []

    detective_data = data["detective"]
    detective = Character(
        name=detective_data["name"],
        role="detective",
        description=detective_data["description"],
        means=detective_data.get("means", "N/A"),
        motive=detective_data.get("motive", "solve the case"),
        opportunity=detective_data.get("opportunity", "arrives after the crime to investigate"),
        alibi=detective_data.get("alibi", "N/A"),
        alibi_is_lie=False,
        is_criminal=False,
        eliminated=False,
        notes=[]
    )
    characters.append(detective)

    criminal_data = data["criminal"]
    criminal = Character(
        name=criminal_data["name"],
        role="criminal",
        description=criminal_data["description"],
        means=criminal_data["means"],
        motive=criminal_data["motive"],
        opportunity=criminal_data["opportunity"],
        alibi=criminal_data["alibi"],
        alibi_is_lie=criminal_data.get("alibi_is_lie", True),
        is_criminal=True,
        eliminated=False,
        notes=[]
    )
    characters.append(criminal)

    for suspect_data in data["suspects"]:
        suspect = Character(
            name=suspect_data["name"],
            role="suspect",
            description=suspect_data["description"],
            means=suspect_data["means"],
            motive=suspect_data["motive"],
            opportunity=suspect_data["opportunity"],
            alibi=suspect_data["alibi"],
            alibi_is_lie=suspect_data.get("alibi_is_lie", False),
            is_criminal=False,
            eliminated=False,
            notes=[]
        )
        characters.append(suspect)

    return characters


# Builds the prompt for the LLM to validate our character list
def build_character_validation_prompt(world: StoryWorld, characters: List[Character]) -> str:
    crime = world.crime_details
    if not crime:
        raise ValueError("No crime details in world.")

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "criminal_name": crime.criminal_name,
            "criminal_motive": crime.criminal_motive,
            "criminal_means": crime.criminal_means,
            "backstory_summary": crime.backstory_summary,
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "is_criminal": c.is_criminal,
            }
            for c in characters
        ]
    }

    return f"""
You are validating the character set for a crime mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Is there exactly 1 detective?
2. Is there exactly 1 criminal and does their name match the hidden criminal_name?
3. Are there exactly 5 non-criminal suspects?
4. Does every suspect have plausible means, motive, or opportunity?
5. Are at least 2 suspects genuinely suspicious at first?
6. Are the characters distinct from one another?
7. Are there any contradictions with the crime setup?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


# Prompts the LLM to validate our character list
def validate_characters(world: StoryWorld, characters: List[Character]) -> dict:
    raw = ask_llm(build_character_validation_prompt(world, characters))
    return extract_json(raw)

In [ ]:
# Builds the prompt for generating clues
def build_clues_prompt(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist before generating clues.")
    if not world.characters:
        raise ValueError("world.characters must exist before generating clues.")

    crime = world.crime_details
    character_payload = [
        {
            "name": c.name,
            "role": c.role,
            "description": c.description,
            "means": c.means,
            "motive": c.motive,
            "opportunity": c.opportunity,
            "alibi": c.alibi,
            "alibi_is_lie": c.alibi_is_lie,
            "is_criminal": c.is_criminal,
        }
        for c in world.characters
    ]

    crime_payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "criminal_means": crime.criminal_means,
        "key_evidence_hidden": crime.key_evidence_hidden,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are generating discoverable clues for a crime mystery.

Hidden crime details:
{json.dumps(crime_payload, indent=2)}

Character list:
{json.dumps(character_payload, indent=2)}

Generate 8 clues.

Return ONLY valid JSON with exactly this structure:
{{
  "clues": [
    {{
      "clue_id": 1,
      "description": "...",
      "location": "...",
      "is_red_herring": false,
      "points_to": "...",
      "why_it_matters": "..."
    }}
  ]
}}

Requirements:
- Generate exactly 8 clues.
- Clues must follow naturally from the true crime and the characters.
- At least 2 clues must initially point toward innocent suspects.
- At least 2 clues must meaningfully connect to the real criminal.
- At least 1 clue should relate to a false alibi.
- At least 1 clue should be ambiguous until later interpretation.
- The clues should not instantly solve the crime when seen alone.
- "points_to" should name a character or a specific event/fact.
- clue_id values must be 1 through 8.

Return JSON only.
""".strip()


# Prompts the LLM and extracts clue data to our data structure
def generate_clues(world: StoryWorld) -> List[Clue]:
    raw = ask_llm(build_clues_prompt(world))
    data = extract_json(raw)

    clues = []
    for clue_data in data["clues"]:
        clue = Clue(
            clue_id=clue_data["clue_id"],
            description=clue_data["description"],
            location=clue_data["location"],
            discovered=False,
            is_red_herring=clue_data.get("is_red_herring", False),
            points_to=clue_data.get("points_to", ""),
            revealed_in_plot_point=-1
        )
        clues.append(clue)

    return clues


# Builds the prompt for the LLM to validate our clues
def build_clue_validation_prompt(world: StoryWorld, clues: List[Clue]) -> str:
    crime = world.crime_details
    if not crime:
        raise ValueError("No crime details in world.")

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "criminal_name": crime.criminal_name,
            "criminal_motive": crime.criminal_motive,
            "criminal_means": crime.criminal_means,
            "backstory_summary": crime.backstory_summary,
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "is_criminal": c.is_criminal,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
            }
            for c in world.characters
        ],
        "clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in clues
        ]
    }

    return f"""
You are validating the clue set for a crime mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Are there exactly 8 clues?
2. Do the clues arise naturally from the crime and character setup?
3. Do at least 2 clues point toward innocent suspects at first?
4. Do at least 2 clues connect meaningfully to the real criminal?
5. Is the mystery neither trivial nor impossible?
6. Are any clues redundant, contradictory, or disconnected?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


# Prompts the LLM to validate our clue list
def validate_clues(world: StoryWorld, clues: List[Clue]) -> dict:
    raw = ask_llm(build_clue_validation_prompt(world, clues))
    return extract_json(raw)

In [ ]:
# Builds the prompt to generate a countdown deadline and consequence for our solving story countdown mechanism
def build_countdown_prompt(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist first")

    crime = world.crime_details

    payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are designing a countdown mechanism for a suspenseful detective story.

The countdown must:
- Be concrete and time-bound
- Be directly connected to the crime or investigation
- Create urgency that increases over time
- Make failure meaningful and irreversible

Crime context:
{json.dumps(payload, indent=2)}

Generate:
1. A specific countdown deadline (what event is approaching?)
2. A specific consequence if the detective fails before the deadline

Constraints:
- The deadline should feel natural to the scenario (not arbitrary)
- The consequence must be severe and story-relevant
- The consequence should make the truth harder or impossible to recover
- Avoid vague phrasing like "things get worse"

Examples of good countdowns:
- "By sunrise, the crime scene will be destroyed by the incoming storm."
- "In 24 hours, the prosecutor will formally charge the wrong suspect."
- "The suspect is scheduled to leave the country tonight."
- "A building demolition will erase crucial evidence."

Do not copy these exactly. Generate a new one specific to the crime.

Return ONLY valid JSON:
{{
  "countdown_deadline": "...",
  "countdown_consequence": "..."
}}
""".strip()


# Prompts the LLM and extracts data to StoryWorld data structure
def generate_countdown(world: StoryWorld) -> StoryWorld:
    raw = ask_llm(build_countdown_prompt(world))
    data = extract_json(raw)

    world.countdown_deadline = data["countdown_deadline"]
    world.countdown_consequence = data["countdown_consequence"]

    return world

In [ ]:
# Function to populate StoryWorld data structure with characters, clues, and countdown mechanism
# Also runs validation functions of our generated characters and clues and outputs if the LLM detects an issue
def populate_story_world(world: StoryWorld) -> StoryWorld:
    if not world.crime_details:
        raise ValueError("Generate crime details first.")

    characters = generate_characters(world)
    char_validation = validate_characters(world, characters)
    if not char_validation["is_valid"]:
        print("Invalid characters -- try again")
        pprint(char_validation)

    world.characters = characters

    clues = generate_clues(world)
    clue_validation = validate_clues(world, clues)
    if not clue_validation["is_valid"]:
        print("Invalid clues -- try again")
        pprint(clue_validation)

    world.clues = clues

    world = generate_countdown(world)
    return world

In [ ]:
def print_characters(world: StoryWorld):
    print("\nCHARACTERS")
    print("=" * 60)
    for c in world.characters:
        print(f"Name: {c.name}")
        print(f"Role: {c.role}")
        print(f"Description: {c.description}")
        print(f"Means: {c.means}")
        print(f"Motive: {c.motive}")
        print(f"Opportunity: {c.opportunity}")
        print(f"Alibi: {c.alibi}")
        print(f"Alibi is lie: {c.alibi_is_lie}")
        print(f"Is criminal: {c.is_criminal}")
        print("-" * 60)


def print_clues(world: StoryWorld):
    print("\nCLUES")
    print("=" * 60)
    for cl in sorted(world.clues, key=lambda x: x.clue_id):
        print(f"Clue #{cl.clue_id}")
        print(f"Description: {cl.description}")
        print(f"Location: {cl.location}")
        print(f"Points to: {cl.points_to}")
        print(f"Red herring: {cl.is_red_herring}")
        print("-" * 60)


def print_countdown(world: StoryWorld):
    print("\nCOUNTDOWN SETUP")
    print("=" * 60)
    print(f"Deadline: {world.countdown_deadline}")
    print(f"Consequence: {world.countdown_consequence}")

In [ ]:
# Runner cell to populate story world with characters, clues, and countdown mechanism
world = populate_story_world(world)

print_characters(world)
print_clues(world)
print_countdown(world)


CHARACTERS
Name: Inspector Vivian Holloway
Role: detective
Description: Sharp-witted and methodical, Inspector Holloway of the Metropolitan Police's Serious Crime Unit has a reputation for her uncanny ability to unravel complex cases by focusing on the overlooked details. She possesses a calm demeanor that belies a fierce intellect, often asking the quietest questions that yield the loudest answers. She's known for her piercing gaze and her reliance on psychological profiling as much as forensic evidence.
Means: N/A
Motive: solve the case
Opportunity: arrives after the crime to investigate
Alibi: N/A
Alibi is lie: False
Is criminal: False
------------------------------------------------------------
Name: Eleanor Vance
Role: criminal
Description: Alistair Finch's immaculately presented and seemingly indispensable personal assistant for over a decade. She is known for her efficiency, discretion, and unwavering loyalty to Finch, managing his intricate schedule and philanthropic ventures 

## Phase 3: Iterative Loop / Meta Controller

In [ ]:
# Helper function for countdown mechanism
def build_countdown_text(step_index: int, total_steps: int = 15) -> str:
    remaining = total_steps - step_index + 1

    if remaining > 10:
        urgency = "low but rising"
    elif remaining > 5:
        urgency = "moderate and increasingly serious"
    elif remaining > 2:
        urgency = "high"
    else:
        urgency = "extreme"

    return (
        f"COUNTDOWN: The detective has only {remaining} major investigative moves remaining "
        f"before the case reaches a point of no return. Urgency level: {urgency}."
    )

In [ ]:
# Summarizes plot so far to pass as context to next LLM prompt
def build_plot_context(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details is missing")

    crime = world.crime_details

    detective = next((c for c in world.characters if c.role == "detective"), None)
    if detective is None:
        raise ValueError("No detective found in world.characters")

    suspects = [c for c in world.characters if c.role in ("suspect", "criminal")]
    discovered_clues = world.discovered_clues()

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "backstory_summary": crime.backstory_summary,
        },
        "detective": {
            "name": detective.name,
            "description": detective.description,
            "motive": detective.motive,
        },
        "active_suspects": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "eliminated": c.eliminated,
            }
            for c in suspects if not c.eliminated
        ],
        "discovered_clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in discovered_clues
        ],
        "undiscovered_clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in world.clues if not cl.discovered
        ],
        "prior_plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in world.plot_points
        ]
    }

    return json.dumps(payload, indent=2)

In [ ]:
# Prompt for generating next plot point
def build_next_plot_point_prompt(world: StoryWorld, step_index: int, total_steps: int = 15) -> str:
    detective = next((c for c in world.characters if c.role == "detective"), None)
    if detective is None:
        raise ValueError("No detective found")

    countdown_text = build_countdown_text(step_index, total_steps)

    detective_goal = (
        f"{detective.name} must identify the true criminal behind the "
        f"{world.crime_details.crime_type} of {world.crime_details.victim} before the final countdown expires."
    )

    deadline = getattr(world, "countdown_deadline", "")
    consequence = getattr(world, "countdown_consequence", "")

    countdown_block = f"""
    DEADLINE:
    {deadline}

    CONSEQUENCE:
    {consequence}
    """

    return f"""
You are the suspense-planning controller for a crime mystery.

Your job is to generate exactly ONE next plot point in a 15-step detective story.

This story must use suspense generation:
1. The audience cares about the detective and the victim.
2. The detective has a concrete goal.
3. Failure has dire consequences.
4. Each new plot point makes success harder, more urgent, or less likely.
5. Suspense comes from narrowing options, increasing uncertainty, and reducing chances to avert failure.
6. Do NOT resolve the full case early.

Detective goal:
{detective_goal}

{countdown_block}

{countdown_text}

Current story state:
{build_plot_context(world)}

Rules for this next plot point:
- This is plot point #{step_index} of {total_steps}.
- The detective must take a concrete action.
- That action must produce a meaningful outcome, but not fully solve the case unless this is the final plot point.
- Introduce or intensify an obstacle.
- Suspense must generally rise over time.
- Reveal 0, 1, or 2 clues at this step.
- Use already discovered clues when relevant.
- A clue can be revealed only once.
- You may eliminate at most one suspect in a single step.
- The detective should not identify the real criminal before plot point 13.
- Plot points 13-15 should feel like endgame escalation.
- A false lead, contradiction, missing evidence, time pressure, or witness unreliability can be used as obstacles.
- Keep the action grounded in crime solving, not action-movie spectacle.

Return ONLY valid JSON with exactly this structure:
{{
  "index": {step_index},
  "title": "...",
  "description": "...",
  "action_taken": "...",
  "outcome": "...",
  "obstacle": "...",
  "suspense_score": 0.0,
  "characters_involved": ["..."],
  "clues_revealed": [1, 2],
  "clues_used": [1, 2],
  "suspect_eliminated": "Name or empty string",
  "countdown_note": "How the countdown pressure changed in this step"
}}

Constraints on suspense_score:
- It should usually increase over time.
- Early steps: around 0.25 to 0.45
- Middle steps: around 0.45 to 0.75
- Late steps: around 0.75 to 0.98

Return JSON only.
""".strip()

In [ ]:
# Integrate/apply new plot point to story
def apply_plot_point_to_world(world: StoryWorld, data: dict, step_index: int = -1) -> PlotPoint:
    # Be defensive: the LLM occasionally omits text fields. Fall back to sensible
    # defaults so the pipeline does not crash mid-generation.
    title = (data.get("title") or "").strip() or f"Plot Point {data.get('index', step_index)}"
    description = (data.get("description") or "").strip() or (data.get("outcome") or "").strip() or title
    action_taken = (data.get("action_taken") or "").strip() or "The detective continues investigating."
    outcome = (data.get("outcome") or "").strip() or description
    obstacle = (data.get("obstacle") or "").strip() or "An unresolved question lingers."

    plot_point = PlotPoint(
        index=int(data.get("index", step_index)),
        title=title,
        description=description,
        action_taken=action_taken,
        outcome=outcome,
        obstacle=obstacle,
        suspense_score=float(data.get("suspense_score", 0.5)),
        characters_involved=data.get("characters_involved", []),
        clues_revealed=data.get("clues_revealed", []),
        clues_used=data.get("clues_used", [])
    )

    # Mark newly revealed clues as discovered
    for clue_id in plot_point.clues_revealed:
        clue = world.get_clue(clue_id)
        if clue:
            clue.discovered = True
            clue.revealed_in_plot_point = plot_point.index

    # Eliminate suspect if applicable
    suspect_name = data.get("suspect_eliminated", "").strip()
    if suspect_name:
        character = world.get_character(suspect_name)
        if character and character.role in ("suspect", "criminal") and not character.is_criminal:
            character.eliminated = True
            character.notes.append(f"Eliminated in plot point {plot_point.index}")

    world.plot_points.append(plot_point)
    return plot_point


In [ ]:
# Prompt for validating plot construction
def build_plot_point_validation_prompt(world: StoryWorld, candidate: dict, total_steps: int = 15) -> str:
    return f"""
You are validating one plot point in a suspenseful detective story.

Current story state:
{build_plot_context(world)}

Candidate plot point:
{json.dumps(candidate, indent=2)}

Check:
1. Is the detective taking a concrete investigative action?
2. Does the plot point increase or maintain suspense appropriately?
3. Does it avoid solving the whole case too early?
4. Are clue reveals valid and non-contradictory?
5. Is any suspect elimination justified?
6. Is the obstacle meaningful?
7. Is this consistent with a countdown-based suspense structure?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


def validate_plot_point(world: StoryWorld, candidate: dict) -> dict:
    raw = ask_llm(build_plot_point_validation_prompt(world, candidate))
    return extract_json(raw)

In [ ]:
# Prompts LLM with build_next_plot_point_prompt, validates, and integrates to story world object
def generate_next_plot_point(world: StoryWorld, step_index: int, total_steps: int = 15, max_attempts: int = 3) -> PlotPoint:
    last_candidate = None

    # Validation / retry mechanism
    for attempt in range(max_attempts):
        prompt = build_next_plot_point_prompt(world, step_index, total_steps)
        raw = ask_llm(prompt)
        candidate = extract_json(raw)
        last_candidate = candidate

        validation = validate_plot_point(world, candidate)

        if validation.get("is_valid", False):
            return apply_plot_point_to_world(world, candidate)

    # fallback: use the last candidate even if imperfect
    if last_candidate is None:
        raise ValueError(f"Failed to generate plot point {step_index}")

    return apply_plot_point_to_world(world, last_candidate)

### Meta Controller

In [ ]:
# Meta controller that progresses story by introducing plot points and injecting suspense
def generate_plot_points(world: StoryWorld, total_steps: int = 15) -> StoryWorld:
    if not world.crime_details:
        raise ValueError("Generate crime details first.")
    if not world.characters:
        raise ValueError("Generate characters first.")
    if not world.clues:
        raise ValueError("Generate clues first.")

    world.plot_points = []

    for step_index in range(1, total_steps + 1):
        print(f"Generating plot point {step_index}/{total_steps}...")
        plot_point = generate_next_plot_point(world, step_index, total_steps)
        print(f"  -> {plot_point.title} (suspense={plot_point.suspense_score:.2f})")

    return world

In [ ]:
# Fully validates plot post-generation
def build_full_plot_validation_prompt(world: StoryWorld) -> str:
    payload = {
        "crime_type": world.crime_details.crime_type if world.crime_details else "",
        "victim": world.crime_details.victim if world.crime_details else "",
        "plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in world.plot_points
        ],
        "remaining_active_suspects": [c.name for c in world.active_suspects()],
        "discovered_clues": [cl.clue_id for cl in world.discovered_clues()]
    }

    return f"""
You are validating the full suspense arc of a 15-point detective mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Do the 15 plot points form a coherent progression?
2. Does suspense generally rise?
3. Does the countdown mechanism feel meaningful?
4. Is the case not solved too early?
5. Are the final 3 plot points endgame escalation?
6. Are clue reveals paced well?
7. Are there any repetitive or filler beats?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."],
  "arc_score": 0.0
}}
""".strip()


def validate_full_plot_arc(world: StoryWorld) -> dict:
    raw = ask_llm(build_full_plot_validation_prompt(world))
    return extract_json(raw)

In [ ]:
def print_plot_points(world: StoryWorld):
    print("\nPLOT POINTS")
    print("=" * 80)

    for p in sorted(world.plot_points, key=lambda x: x.index):
        print(f"[{p.index}] {p.title}")
        print(f"Description: {p.description}")
        print(f"Action Taken: {p.action_taken}")
        print(f"Outcome: {p.outcome}")
        print(f"Obstacle: {p.obstacle}")
        print(f"Suspense Score: {p.suspense_score:.2f}")
        print(f"Characters Involved: {p.characters_involved}")
        print(f"Clues Revealed: {p.clues_revealed}")
        print(f"Clues Used: {p.clues_used}")
        print("-" * 80)

In [ ]:
# Runner cell to generate plot points and validate story

world = generate_plot_points(world, total_steps=15)
print_plot_points(world)

arc_validation = validate_full_plot_arc(world)
print("\nFULL ARC VALIDATION")
pprint(arc_validation)

Generating plot point 1/15...
  -> A Looming Threat and a Glimpse into the Victim's Secrets (suspense=0.35)
Generating plot point 2/15...
  -> A Button, A Lie, and Lingering Pressure (suspense=0.40)
Generating plot point 3/15...
  -> A Button's Whisper and a Fleeting Alibi (suspense=0.48)
Generating plot point 4/15...
  -> A Ghostly Trace and the Looming Silence (suspense=0.59)
Generating plot point 5/15...
  -> Ashes of Evidence (suspense=0.71)
Generating plot point 6/15...
  -> A Scorched Truth and a Desperate Deception (suspense=0.81)
Generating plot point 7/15...
  -> A Concocted Truth and a Race for Proof (suspense=0.86)
Generating plot point 8/15...
  -> A Scratched Alibi and a Calculated Calm (suspense=0.90)
Generating plot point 9/15...
  -> A Hidden Toolkit and a Public Gambit (suspense=0.92)
Generating plot point 10/15...
  -> A Shield of Sickness and a Race for Intent (suspense=0.94)
Generating plot point 11/15...
  -> A Written Warning and the Final Hour (suspense=0.95)
Gen

## Phase 4: Validation + Final Narrative Assembly

In [ ]:
# Functions for post-generation consistency check

def build_consistency_check_prompt(world: StoryWorld) -> str:
    payload = {
        "crime_details": {
            "crime_type": world.crime_details.crime_type if world.crime_details else "",
            "victim": world.crime_details.victim if world.crime_details else "",
            "location": world.crime_details.location if world.crime_details else "",
            "time_of_crime": world.crime_details.time_of_crime if world.crime_details else "",
            "method": world.crime_details.method if world.crime_details else "",
            "criminal_name": world.crime_details.criminal_name if world.crime_details else "",
            "criminal_motive": world.crime_details.criminal_motive if world.crime_details else "",
            "criminal_means": world.crime_details.criminal_means if world.crime_details else "",
            "backstory_summary": world.crime_details.backstory_summary if world.crime_details else "",
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "is_criminal": c.is_criminal,
                "eliminated": c.eliminated,
                "notes": c.notes,
            }
            for c in world.characters
        ],
        "clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "discovered": cl.discovered,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
                "revealed_in_plot_point": cl.revealed_in_plot_point,
            }
            for cl in world.clues
        ],
        "plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in sorted(world.plot_points, key=lambda x: x.index)
        ]
    }

    return f"""
You are checking a crime mystery story plan for contradictions and logic problems.

Review the following story materials:
{json.dumps(payload, indent=2)}

Check for:
1. Character contradictions
2. Clue contradictions
3. Impossible timeline issues
4. Plot point outcomes that conflict with later events
5. Eliminated suspects acting as if still viable later without explanation
6. Clues that appear without being revealed
7. Any mismatch between the hidden crime and the detective's reconstructed story
8. Whether the final solution points to the real criminal

Return ONLY valid JSON in exactly this format:
{{
  "is_consistent": true,
  "issues": [
    "...",
    "..."
  ],
  "summary": "Brief overall judgment."
}}
""".strip()


def run_consistency_check(world: StoryWorld) -> dict:
    raw = ask_llm(build_consistency_check_prompt(world))
    return extract_json(raw)

In [ ]:
# Run consistency check on world state
consistency_report = run_consistency_check(world)
print("CONSISTENCY CHECK")
pprint(consistency_report)

CONSISTENCY CHECK
{'is_consistent': True,
 'issues': ["Clues 3 and 8 are designated as 'red herrings' but have "
            "'revealed_in_plot_point: -1' and 'discovered: false'. This "
            'implies they are never actually discovered or presented to '
            'Inspector Holloway within the narrative. While not a '
            'contradiction, it means these clues do not function as active red '
            "herrings for the detective's investigation, only as information "
            'known to the reader. If intended to mislead the detective, they '
            'should be discovered at a relevant plot point.',
            'The timeline in the latter plot points (PP10-PP14) is extremely '
            'compressed, with crucial forensic analyses and legal maneuvers '
            'occurring within minutes leading up to a 2:30 AM deadline. While '
            'designed to heighten suspense, it pushes the boundaries of '
            'realistic operational speed for forensic scien

In [ ]:
# Functions for building narrative/prose from generated story world object
def build_narrative_assembly_prompt(world: StoryWorld) -> str:
    detective = next((c for c in world.characters if c.role == "detective"), None)

    characters_payload = [
        {
            "name": c.name,
            "role": c.role,
            "description": c.description,
            "motive": c.motive,
            "opportunity": c.opportunity,
            "alibi": c.alibi,
            "is_criminal": c.is_criminal
        }
        for c in world.characters
    ]

    clues_payload = [
        {
            "clue_id": cl.clue_id,
            "description": cl.description,
            "location": cl.location,
            "is_red_herring": cl.is_red_herring,
            "points_to": cl.points_to,
            "revealed_in_plot_point": cl.revealed_in_plot_point
        }
        for cl in world.clues
    ]

    plot_descriptions = [
        {
            "index": p.index,
            "title": p.title,
            "description": p.description,
            "action_taken": p.action_taken,
            "outcome": p.outcome,
            "obstacle": p.obstacle,
            "characters_involved": p.characters_involved,
            "clues_revealed": p.clues_revealed,
            "clues_used": p.clues_used
        }
        for p in sorted(world.plot_points, key=lambda x: x.index)
    ]

    payload = {
        "detective_name": detective.name if detective else "The detective",
        "crime_type": world.crime_details.crime_type if world.crime_details else "",
        "victim": world.crime_details.victim if world.crime_details else "",
        "location": world.crime_details.location if world.crime_details else "",
        "countdown_deadline": getattr(world, "countdown_deadline", ""),
        "countdown_consequence": getattr(world, "countdown_consequence", ""),
        "characters": characters_payload,
        "clues": clues_payload,
        "plot_points": plot_descriptions
    }

    return f"""
You are a skilled mystery novelist.

Write a complete mystery story based on the materials below.

Story materials:
{json.dumps(payload, indent=2)}

Instructions:
- Write a full, immersive narrative in the style of a mystery novel.
- Do NOT write a synopsis or summary.
- Expand the plot points into scenes with atmosphere, movement, reflection, and tension.
- Preserve the sequence of events and the underlying logic of the investigation.
- Use the characters consistently with their descriptions and roles.
- Weave clues naturally into the prose.
- Develop red herrings in a believable way.
- Maintain strong suspense throughout, with the countdown pressure growing more intense as the story progresses.
- The detective should gradually piece the truth together through observation, inference, and misdirection.
- Use polished, engaging prose and natural transitions.
- The ending should feel like a proper mystery revelation, with the detective identifying the true criminal in a satisfying way.
- A detailed and well-written story is better than a compressed one.
- Please keep it 1500-2000 words.

Literary style:
- Clear but literary prose
- Varied sentence structure
- Strong scene transitions
- Show, don’t just tell
- Dialogue should be used frequently and naturally throughout the story

Dialogue requirements:
- Include regular dialogue between characters (detective, suspects, witnesses)
- Conversations should drive the investigation forward
- Use dialogue to reveal clues, contradictions, and personality
- Avoid long stretches without dialogue
- Dialogue should feel natural, not overly formal or robotic
- Mix dialogue with action and internal thoughts

Do not:
- mention "plot point"
- mention "countdown mechanism"
- list clues explicitly
- write headings, bullet points, or outline labels

Output only the final story.
""".strip()


def assemble_final_narrative(world: StoryWorld) -> str:
    if len(world.plot_points) < 15:
        raise ValueError("Need at least 15 plot points before assembling final narrative.")

    raw = ask_llm(build_narrative_assembly_prompt(world))
    world.final_narrative = raw.strip()
    return world.final_narrative

In [ ]:
# Runs final narrative assembly and prints output

final_story = assemble_final_narrative(world)
print(world.final_narrative)

The Azure Bloom Hotel, usually a shimmering beacon of polished chrome and hushed luxury, now hummed with a different kind of tension. Beneath the glittering chandeliers of the grand lobby, Inspector Vivian Holloway moved with a quiet purpose that belied the frantic energy of the uniformed officers around her. She had arrived at 1:00 AM, the call pulling her from a restless sleep. Alistair Finch, 62, renowned art collector, found dead in his private lounge. Initial reports whispered "cardiac event," but Holloway’s gut, a reliable compass in her line of work, pointed towards something far more sinister.

“Inspector Holloway,” Sergeant Miller greeted, his face etched with exhaustion. “Forensics is processing the lounge. Hotel management is… less than thrilled. Their ‘discreet incident cleanup’ protocol is scheduled to begin at 2:30 AM for the lounge, then sweep the rest of the hotel. They’re adamant about minimising disruption.”

Holloway’s eyes narrowed, a glint of steel beneath her calm

# Phase II: Search-Based Drama Manager

Phase 1 above produces a finished, linear story. Phase II reuses everything in `world` (crime, characters, clues, plot points, countdown) and turns it into a **playable, text-based interactive mystery** governed by a search-based drama manager (DM).

Pipeline (matches the Phase II project specification):
1. **Plot Graph** — convert the 15 linear plot points into an abstract DAG of nodes with preconditions and effects.
2. **World Generation** — generate locations, items, and NPC placements that support every node's preconditions.
3. **World State** — runtime state that is cheap to clone for DM rollouts.
4. **Action Parser** — natural-language input → structured `(verb, target, indirect)` action.
5. **Game Engine** — validates the action, updates state, fires any plot-graph nodes whose preconditions are now satisfied (constituent / exceptional / neutral classification).
6. **Drama Manager** — sampling-based rollouts over candidate interventions (`hint`, `temp_deny`, `cause`, `glow`, `noop`); picks the move that maximises a utility combining solvability, pacing, suspense, and (negative) manipulation.
7. **Intervention Rendering** — translate the chosen DM move into diegetic prose.
8. **Game Loop** — interactive REPL.
9. **Evaluation Harness** — non-interactive auto-play with metrics from the project plan.

## Step 1 — Plot Graph

Each linear plot point becomes a `PlotNode` with explicit `preconditions` and `effects`. Predicates use a tiny vocabulary so the engine can check them deterministically:

- `["clue_discovered", <id>]`
- `["interviewed", <name>]`
- `["at_location", <name>]`
- `["suspect_eliminated", <name>]`
- `["item_obtained", <name>]`
- `["flag", <name>]`

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple, Callable
import copy
import random


@dataclass
class PlotNode:
    """One node in the abstract plot graph."""
    id: int
    title: str
    description: str
    preconditions: List[List[Any]] = field(default_factory=list)
    effects: List[List[Any]] = field(default_factory=list)
    is_terminal: bool = False  # True if this node represents the case being solved


def build_plot_graph_prompt(world: StoryWorld) -> str:
    plot_payload = [
        {
            "index": p.index,
            "title": p.title,
            "description": p.description,
            "action_taken": p.action_taken,
            "outcome": p.outcome,
            "characters_involved": p.characters_involved,
            "clues_revealed": p.clues_revealed,
            "clues_used": p.clues_used,
        }
        for p in sorted(world.plot_points, key=lambda x: x.index)
    ]
    detective = next((c.name for c in world.characters if c.role == "detective"), "Detective")
    suspects = [c.name for c in world.characters if c.role in ("suspect", "criminal")]

    return f"""
You are converting a linear mystery story into an abstract plot graph for an interactive game.

Each plot point becomes a node. Each node has:
- preconditions: facts that must be true before the node can fire
- effects: facts that become true once the node fires

Use ONLY the following predicate types:
- ["clue_discovered", <clue_id:int>]
- ["interviewed", <character_name:str>]
- ["at_location", <location_name:str>]
- ["suspect_eliminated", <character_name:str>]
- ["item_obtained", <item_name:str>]
- ["flag", <flag_name:str>]

Detective: {detective}
Suspects: {suspects}

Plot points:
{json.dumps(plot_payload, indent=2)}

Return ONLY valid JSON:
{{
  "nodes": [
    {{
      "id": 1,
      "title": "...",
      "description": "...",
      "preconditions": [["clue_discovered", 1]],
      "effects": [["clue_discovered", 3], ["interviewed", "Marcus"]],
      "is_terminal": false
    }}
  ]
}}

Rules:
- Node ids should equal the original plot-point indices.
- The first node should typically have no preconditions.
- Effects of node N must include any clues_revealed in plot point N.
- Preconditions of node N should reference effects of earlier nodes when the original story used them.
- Mark exactly one final node (where the case is solved) as is_terminal=true.
- Stay grounded in the existing story; do not invent new clues or characters.
""".strip()


def extract_plot_graph(world: StoryWorld) -> List[PlotNode]:
    raw = ask_llm(build_plot_graph_prompt(world))
    data = extract_json(raw)
    nodes = []
    for n in data["nodes"]:
        nodes.append(PlotNode(
            id=int(n["id"]),
            title=n["title"],
            description=n["description"],
            preconditions=[list(p) for p in n.get("preconditions", [])],
            effects=[list(e) for e in n.get("effects", [])],
            is_terminal=bool(n.get("is_terminal", False)),
        ))
    if not any(n.is_terminal for n in nodes) and nodes:
        nodes[-1].is_terminal = True
    return nodes


In [ ]:
# Runner: extract the plot graph
world.plot_graph = extract_plot_graph(world)

print(f"Plot graph: {len(world.plot_graph)} nodes")
for n in world.plot_graph:
    flag = " [TERMINAL]" if n.is_terminal else ""
    print(f"[{n.id}] {n.title}{flag}")
    print(f"    pre: {n.preconditions}")
    print(f"    eff: {n.effects}")


Plot graph: 15 nodes
[1] A Looming Threat and a Glimpse into the Victim's Secrets
    pre: [['at_location', 'Azure Bloom Hotel']]
    eff: [['clue_discovered', 1], ['clue_discovered', 5], ['at_location', 'Azure Bloom Hotel']]
[2] A Button, A Lie, and Lingering Pressure
    pre: [['clue_discovered', 1], ['clue_discovered', 5], ['at_location', 'Azure Bloom Hotel']]
    eff: [['clue_discovered', 4], ['interviewed', 'Eleanor Vance']]
[3] A Button's Whisper and a Fleeting Alibi
    pre: [['clue_discovered', 4], ['interviewed', 'Eleanor Vance'], ['at_location', 'Azure Bloom Hotel']]
    eff: [['clue_discovered', 2]]
[4] A Ghostly Trace and the Looming Silence
    pre: [['clue_discovered', 2], ['at_location', 'Azure Bloom Hotel']]
    eff: [['clue_discovered', 7]]
[5] Ashes of Evidence
    pre: [['clue_discovered', 2], ['clue_discovered', 7], ['at_location', 'Azure Bloom Hotel']]
    eff: [['flag', 'Eleanor_Vance_Observed_Suspiciously'], ['at_location', 'Hotel_Incinerator_Area']]
[6] A Scorch

## Step 2 — World Generation

Generate the explorable map: locations with adjacency, items (one per clue), and starting positions for the detective and every NPC. The LLM decides spatial layout; we symmetrize adjacency afterwards as a defensive cleanup.

In [ ]:
@dataclass
class Location:
    name: str
    description: str
    neighbors: List[str] = field(default_factory=list)


@dataclass
class Item:
    name: str
    description: str
    location: str
    related_clue_id: int = -1


@dataclass
class GameWorld:
    locations: List[Location] = field(default_factory=list)
    items: List[Item] = field(default_factory=list)
    npc_locations: Dict[str, str] = field(default_factory=dict)
    starting_location: str = ""

    def get_location(self, name: str) -> Optional[Location]:
        for loc in self.locations:
            if loc.name.lower() == name.lower():
                return loc
        return None


def build_world_gen_prompt(world: StoryWorld) -> str:
    crime = world.crime_details
    char_payload = [
        {"name": c.name, "role": c.role, "description": c.description}
        for c in world.characters
    ]
    clue_payload = [
        {"clue_id": cl.clue_id, "description": cl.description, "location": cl.location}
        for cl in world.clues
    ]
    return f"""
You are generating the explorable game world for an interactive crime mystery.

Crime: {crime.crime_type} of {crime.victim} at {crime.location} ({crime.time_of_crime}).

Characters:
{json.dumps(char_payload, indent=2)}

Clues (the 'location' field is the original story description, which you should map to one of your generated locations):
{json.dumps(clue_payload, indent=2)}

Generate:
- 6 to 10 distinct, named locations covering the story setting.
- An undirected adjacency graph (commonsense neighbours; map must be connected).
- One starting location for the detective (typically arrival point or main hall).
- One item per clue; each item is the physical object the player examines to discover the clue, placed in a sensible location.
- A starting location for every non-detective character (suspect / criminal / witness).

Return ONLY valid JSON:
{{
  "locations": [
    {{"name": "...", "description": "...", "neighbors": ["...", "..."]}}
  ],
  "items": [
    {{"name": "...", "description": "...", "location": "...", "related_clue_id": 1}}
  ],
  "npc_locations": {{"Character Name": "Location Name"}},
  "starting_location": "..."
}}

Rules:
- Adjacency must be (mostly) symmetric. If A lists B, B should list A.
- Every location must be reachable from starting_location.
- Every clue id (1..N) must have exactly one corresponding item.
- npc_locations values must reference an existing location name.
- Names must be unique. Keep them short (1-3 words).
""".strip()


def generate_game_world(world: StoryWorld) -> GameWorld:
    raw = ask_llm(build_world_gen_prompt(world))
    data = extract_json(raw)
    gw = GameWorld(
        locations=[
            Location(name=l["name"], description=l["description"], neighbors=list(l.get("neighbors", [])))
            for l in data["locations"]
        ],
        items=[
            Item(
                name=i["name"],
                description=i["description"],
                location=i["location"],
                related_clue_id=int(i.get("related_clue_id", -1)),
            )
            for i in data["items"]
        ],
        npc_locations=dict(data.get("npc_locations", {})),
        starting_location=data.get("starting_location", ""),
    )
    # Symmetrize adjacency
    by_name = {l.name: l for l in gw.locations}
    for loc in gw.locations:
        for n in list(loc.neighbors):
            if n in by_name and loc.name not in by_name[n].neighbors:
                by_name[n].neighbors.append(loc.name)
    return gw


In [ ]:
# Runner: generate the game world
world.game_world = generate_game_world(world)

gw = world.game_world
print(f"Locations: {len(gw.locations)} | Items: {len(gw.items)} | NPCs placed: {len(gw.npc_locations)}")
print(f"Start: {gw.starting_location}\n")
print("Map:")
for loc in gw.locations:
    print(f"  - {loc.name}: {loc.neighbors}")
print("\nItems:")
for it in gw.items:
    print(f"  - {it.name} (clue {it.related_clue_id}) @ {it.location}")
print("\nNPCs:")
for name, loc in gw.npc_locations.items():
    print(f"  - {name} @ {loc}")


Locations: 8 | Items: 8 | NPCs placed: 6
Start: Azure Bloom Lobby

Map:
  - Azure Bloom Lobby: ['Grand Ballroom', 'Hotel Management Office']
  - Grand Ballroom: ['Azure Bloom Lobby', "Finch's Private Lounge", 'Service Corridor']
  - Finch's Private Lounge: ['Grand Ballroom', 'Service Corridor']
  - Service Corridor: ['Grand Ballroom', "Finch's Private Lounge", 'Hotel Security Office', 'Staff Laundry Room', 'Kitchen & Prep Area']
  - Hotel Security Office: ['Service Corridor']
  - Staff Laundry Room: ['Service Corridor']
  - Hotel Management Office: ['Azure Bloom Lobby']
  - Kitchen & Prep Area: ['Service Corridor']

Items:
  - Security Log Terminal (clue 1) @ Hotel Security Office
  - Event Coordinator's Notepad (clue 2) @ Grand Ballroom
  - Corridor Security Footage (clue 3) @ Hotel Security Office
  - Ornate Pearl Button (clue 4) @ Finch's Private Lounge
  - Alistair Finch's Tablet (clue 5) @ Finch's Private Lounge
  - Finch's Custom Vape Pen (clue 6) @ Finch's Private Lounge
  - Pun

## Step 3 — World State

`WorldState` is the runtime mutable state for one play-through. It is intentionally a flat dataclass of basic types so `copy.deepcopy` is cheap (the DM rollouts will fork it many times per turn). Predicate evaluation and effect application are pure functions over this state.

In [ ]:
@dataclass
class WorldState:
    """Runtime state for a single play-through. Cheap to clone for DM rollouts."""
    player_location: str = ""
    inventory: List[str] = field(default_factory=list)
    discovered_clues: List[int] = field(default_factory=list)
    interviewed: List[str] = field(default_factory=list)
    eliminated_suspects: List[str] = field(default_factory=list)
    flags: Dict[str, bool] = field(default_factory=dict)
    executed_nodes: List[int] = field(default_factory=list)
    perm_denied_nodes: List[int] = field(default_factory=list)
    temp_denied_nodes: Dict[int, int] = field(default_factory=dict)  # node_id -> turn it expires
    turn: int = 0
    intervention_count: int = 0
    case_solved: bool = False
    last_classification: str = ""


def predicate_satisfied(state: WorldState, pred: List[Any]) -> bool:
    if not pred:
        return True
    kind = pred[0]
    arg = pred[1] if len(pred) > 1 else None
    if kind == "clue_discovered":
        return int(arg) in state.discovered_clues
    if kind == "interviewed":
        return arg in state.interviewed
    if kind == "at_location":
        return state.player_location == arg
    if kind == "suspect_eliminated":
        return arg in state.eliminated_suspects
    if kind == "item_obtained":
        return arg in state.inventory
    if kind == "flag":
        return bool(state.flags.get(arg, False))
    return False


def apply_effect(state: WorldState, eff: List[Any]) -> None:
    if not eff:
        return
    kind = eff[0]
    arg = eff[1] if len(eff) > 1 else None
    if kind == "clue_discovered":
        cid = int(arg)
        if cid not in state.discovered_clues:
            state.discovered_clues.append(cid)
    elif kind == "interviewed" and arg not in state.interviewed:
        state.interviewed.append(arg)
    elif kind == "suspect_eliminated" and arg not in state.eliminated_suspects:
        state.eliminated_suspects.append(arg)
    elif kind == "item_obtained" and arg not in state.inventory:
        state.inventory.append(arg)
    elif kind == "flag":
        state.flags[arg] = True


def initialize_state(world: StoryWorld) -> WorldState:
    return WorldState(player_location=world.game_world.starting_location)


def clone_state(state: WorldState) -> WorldState:
    return copy.deepcopy(state)


## Step 4 — Action Parser

`parse_action` turns the player's free-text input into a structured `Action` using the LLM. The prompt is grounded in the *current* world state so the parser only proposes actions that reference visible items, neighbouring rooms, NPCs in the room, etc.

In [ ]:
@dataclass
class Action:
    verb: str = "unknown"      # move | examine | interview | ask_about | use_item | accuse | look | unknown
    target: str = ""
    indirect: str = ""
    raw: str = ""


def state_summary_for_parser(world: StoryWorld, state: WorldState) -> str:
    here = world.game_world.get_location(state.player_location)
    items_here = [it.name for it in world.game_world.items if it.location == state.player_location]
    npcs_here = [name for name, loc in world.game_world.npc_locations.items() if loc == state.player_location]
    suspects = [c.name for c in world.characters
                if c.role in ("suspect", "criminal") and c.name not in state.eliminated_suspects]
    return json.dumps({
        "current_location": state.player_location,
        "neighbors": here.neighbors if here else [],
        "items_here": items_here,
        "npcs_here": npcs_here,
        "inventory": state.inventory,
        "interviewed_so_far": state.interviewed,
        "discovered_clue_ids": state.discovered_clues,
        "remaining_active_suspects": suspects,
    }, indent=2)


def parse_action(world: StoryWorld, state: WorldState, nl_text: str) -> Action:
    prompt = f"""
You are an action parser for a text-based mystery game.

Map the player\'s natural-language input into ONE structured action.

Allowed verbs:
- "move"      target = a neighbouring location name
- "examine"   target = an item name in the current location
- "interview" target = an NPC name (must be in the same location)
- "ask_about" target = NPC name; indirect = topic / clue / suspect
- "use_item"  target = inventory item; indirect = optional target of use
- "accuse"    target = suspect name (the player\'s final answer)
- "look"      no target (player surveys the room)
- "unknown"   when nothing matches

Current state:
{state_summary_for_parser(world, state)}

Player input: {nl_text!r}

Return ONLY valid JSON: {{"verb": "...", "target": "...", "indirect": ""}}
""".strip()
    try:
        raw = ask_llm(prompt)
        data = extract_json(raw)
        return Action(
            verb=data.get("verb", "unknown"),
            target=data.get("target", "") or "",
            indirect=data.get("indirect", "") or "",
            raw=nl_text,
        )
    except Exception:
        return Action(verb="unknown", raw=nl_text)


## Step 5 — Game Engine (trigger & validation)

`GameEngine.step` is the heart of the loop:

1. Validate the action against the current state.
2. Apply the action's direct effects (move location, mark a clue discovered when the right item is examined, mark an NPC interviewed, etc.).
3. Walk the plot graph; any pending node whose preconditions are now satisfied (and which is not denied by the DM) fires and applies its own effects.
4. Classify the action: `constituent` if it advanced a node, `case_solved` / `false_accusation` for accusations, otherwise `neutral` or `invalid`.

In [ ]:
class GameEngine:
    def __init__(self, world: StoryWorld):
        self.world = world

    def _is_legal(self, state: WorldState, action: Action) -> Tuple[bool, str]:
        gw = self.world.game_world
        if action.verb == "move":
            here = gw.get_location(state.player_location)
            if not here or action.target not in here.neighbors:
                return False, f"You can\'t go to {action.target!r} from here."
        elif action.verb == "examine":
            it = next((i for i in gw.items if i.name == action.target and i.location == state.player_location), None)
            if not it:
                return False, f"There is no {action.target!r} here to examine."
        elif action.verb == "interview":
            if gw.npc_locations.get(action.target) != state.player_location:
                return False, f"{action.target} is not in this room."
            if action.target in state.interviewed:
                return False, f"You\'ve already interviewed {action.target}."
        elif action.verb == "ask_about":
            if gw.npc_locations.get(action.target) != state.player_location:
                return False, f"{action.target} is not here to question."
        elif action.verb == "accuse":
            names = [c.name for c in self.world.characters if c.role in ("suspect", "criminal")]
            if action.target not in names:
                return False, f"{action.target} is not a known suspect."
        elif action.verb == "unknown":
            return False, "I didn\'t understand that. Try \'examine X\', \'move to Y\', \'interview Z\', or \'accuse W\'."
        return True, ""

    def _apply_action_effects(self, state: WorldState, action: Action) -> None:
        gw = self.world.game_world
        if action.verb == "move":
            state.player_location = action.target
        elif action.verb == "examine":
            it = next((i for i in gw.items if i.name == action.target and i.location == state.player_location), None)
            if it and it.related_clue_id > 0:
                apply_effect(state, ["clue_discovered", it.related_clue_id])
        elif action.verb == "interview":
            apply_effect(state, ["interviewed", action.target])

    def _trigger_nodes(self, state: WorldState) -> List[int]:
        fired: List[int] = []
        # Iterate to a fixed point so chained nodes fire in one engine step
        progressed = True
        while progressed:
            progressed = False
            for node in self.world.plot_graph:
                if node.id in state.executed_nodes or node.id in state.perm_denied_nodes:
                    continue
                if node.id in state.temp_denied_nodes and state.turn < state.temp_denied_nodes[node.id]:
                    continue
                if all(predicate_satisfied(state, p) for p in node.preconditions):
                    for eff in node.effects:
                        apply_effect(state, eff)
                    state.executed_nodes.append(node.id)
                    fired.append(node.id)
                    if node.is_terminal:
                        state.case_solved = True
                    progressed = True
        return fired

    def step(self, state: WorldState, action: Action) -> Dict[str, Any]:
        state.turn += 1
        legal, msg = self._is_legal(state, action)
        if not legal:
            state.last_classification = "invalid"
            return {"ok": False, "message": msg, "fired_nodes": [], "classification": "invalid"}

        # Accusation is a terminal verb handled separately
        if action.verb == "accuse":
            criminal = next((c.name for c in self.world.characters if c.is_criminal), "")
            min_clues_needed = max(3, len(self.world.clues) // 2)
            if action.target == criminal and len(state.discovered_clues) >= min_clues_needed:
                state.case_solved = True
                state.last_classification = "case_solved"
                return {"ok": True, "message": "", "fired_nodes": [], "classification": "case_solved", "case_solved": True}
            else:
                state.last_classification = "false_accusation"
                return {"ok": True, "message": "", "fired_nodes": [], "classification": "false_accusation", "case_solved": False}

        self._apply_action_effects(state, action)
        fired = self._trigger_nodes(state)
        classification = "constituent" if fired else "neutral"
        state.last_classification = classification
        return {
            "ok": True,
            "message": "",
            "fired_nodes": fired,
            "classification": classification,
            "case_solved": state.case_solved,
        }


## Step 6 — Drama Manager (search & scoring)

The DM enumerates a small set of candidate interventions, simulates a few short rollouts forward (using a heuristic player model — *not* an LLM, to avoid search explosion as flagged on slide 9), scores the resulting states, and picks the argmax.

**Candidate moves** map directly to the project spec:

- `noop` — let the player drive
- `hint(clue_id)` — diegetic nudge toward an undiscovered clue
- `glow(item)` — make an in-room item conspicuous
- `temp_deny(node, ttl)` — block a node from firing for a few turns (pacing)
- `cause(node)` — fire a near-ready node early (avert dead ends)

**Scoring** combines slide 8's four criteria:

- `+ solvability`  — a path to the terminal node still exists
- `+ pacing`       — node-completion progress tracks turn count
- `+ suspense`     — reward clue discovery
- `− manipulation` — soft cap on number of interventions used

In [ ]:
@dataclass
class Intervention:
    kind: str           # noop | hint | glow | temp_deny | perm_deny | cause
    target: Any = None
    ttl_turns: int = 0


def candidate_interventions(world: StoryWorld, state: WorldState) -> List[Intervention]:
    cands: List[Intervention] = [Intervention(kind="noop")]

    # Hints toward undiscovered clues
    for cl in world.clues:
        if cl.clue_id not in state.discovered_clues:
            cands.append(Intervention(kind="hint", target=cl.clue_id))

    # Glow: highlight an item in the player\'s current room with an undiscovered clue
    for it in world.game_world.items:
        if (it.related_clue_id > 0
                and it.related_clue_id not in state.discovered_clues
                and it.location == state.player_location):
            cands.append(Intervention(kind="glow", target=it.name))

    # temp_deny: any currently fire-able node — keeps the case from resolving too fast
    for node in world.plot_graph:
        if node.id in state.executed_nodes or node.id in state.perm_denied_nodes:
            continue
        if all(predicate_satisfied(state, p) for p in node.preconditions):
            if not node.is_terminal:  # never deny the win condition
                cands.append(Intervention(kind="temp_deny", target=node.id, ttl_turns=3))

    # cause: nudge a near-ready node forward (≤ 1 missing precondition)
    for node in world.plot_graph:
        if node.id in state.executed_nodes or node.id in state.perm_denied_nodes:
            continue
        missing = [p for p in node.preconditions if not predicate_satisfied(state, p)]
        if 0 < len(missing) <= 1 and not node.is_terminal:
            cands.append(Intervention(kind="cause", target=node.id))

    return cands


def apply_intervention(world: StoryWorld, state: WorldState, iv: Intervention) -> None:
    if iv.kind == "noop":
        return
    state.intervention_count += 1
    if iv.kind == "temp_deny":
        state.temp_denied_nodes[int(iv.target)] = state.turn + iv.ttl_turns
    elif iv.kind == "perm_deny":
        if int(iv.target) not in state.perm_denied_nodes:
            state.perm_denied_nodes.append(int(iv.target))
    elif iv.kind == "cause":
        node = next((n for n in world.plot_graph if n.id == int(iv.target)), None)
        if node and node.id not in state.executed_nodes:
            for eff in node.effects:
                apply_effect(state, eff)
            state.executed_nodes.append(node.id)
            if node.is_terminal:
                state.case_solved = True
    # hint / glow are pure narration


def solvable(world: StoryWorld, state: WorldState) -> bool:
    """At least one non-denied terminal node exists."""
    terminals = [n for n in world.plot_graph if n.is_terminal]
    if not terminals:
        return True
    return any(t.id not in state.perm_denied_nodes for t in terminals)


def score_state(world: StoryWorld, state: WorldState, intervention_budget: int = 8) -> float:
    score = 0.0
    score += 1.0 if solvable(world, state) else -2.0
    progress = len(state.executed_nodes) / max(1, len(world.plot_graph))
    expected = min(1.0, state.turn / 20.0)
    score += 1.0 - abs(progress - expected)
    score += 0.1 * len(state.discovered_clues)
    score -= 0.25 * max(0, state.intervention_count - intervention_budget)
    if state.case_solved:
        score += 2.0
    return score


def player_model_action(world: StoryWorld, state: WorldState) -> Action:
    """Heuristic player used inside DM rollouts and as a baseline auto-player."""
    gw = world.game_world
    # Prefer examining an item with an undiscovered clue here
    for it in gw.items:
        if it.location == state.player_location and it.related_clue_id not in state.discovered_clues:
            return Action(verb="examine", target=it.name)
    # Then interview a not-yet-interviewed NPC here
    for name, loc in gw.npc_locations.items():
        if loc == state.player_location and name not in state.interviewed:
            return Action(verb="interview", target=name)
    # If we have enough clues, attempt accusation against the most-implicated suspect
    min_clues_needed = max(3, len(world.clues) // 2)
    if len(state.discovered_clues) >= min_clues_needed:
        criminal = next((c.name for c in world.characters if c.is_criminal), None)
        if criminal:
            return Action(verb="accuse", target=criminal)
    # Otherwise wander
    here = gw.get_location(state.player_location)
    if here and here.neighbors:
        return Action(verb="move", target=random.choice(here.neighbors))
    return Action(verb="look")


def rollout(world: StoryWorld, engine: "GameEngine", start: WorldState, depth: int = 5) -> WorldState:
    state = clone_state(start)
    for _ in range(depth):
        if state.case_solved:
            break
        action = player_model_action(world, state)
        engine.step(state, action)
    return state


def drama_manager_decide(
    world: StoryWorld,
    engine: "GameEngine",
    state: WorldState,
    num_samples: int = 3,
    depth: int = 4,
) -> Intervention:
    cands = candidate_interventions(world, state)
    best_iv = cands[0]
    best_score = float("-inf")
    for iv in cands:
        avg = 0.0
        for _ in range(num_samples):
            sim = clone_state(state)
            apply_intervention(world, sim, iv)
            end = rollout(world, engine, sim, depth=depth)
            avg += score_state(world, end)
        avg /= num_samples
        if avg > best_score:
            best_score = avg
            best_iv = iv
    return best_iv


## Step 7 — Intervention rendering

The DM picks a *structured* move; this step turns it into in-world prose. A small LLM call wraps the chosen intervention with sensory or atmospheric language. The renderer is told never to mention game-engine concepts.

In [ ]:
def render_intervention(world: StoryWorld, state: WorldState, iv: Intervention) -> str:
    if iv.kind == "noop":
        return ""
    payload: Dict[str, Any] = {"kind": iv.kind, "target": iv.target}
    if iv.kind == "hint":
        clue = world.get_clue(int(iv.target))
        payload["clue_description"] = clue.description if clue else ""
    elif iv.kind == "glow":
        item = next((it for it in world.game_world.items if it.name == iv.target), None)
        payload["item_description"] = item.description if item else ""
    elif iv.kind in ("temp_deny", "perm_deny", "cause"):
        node = next((n for n in world.plot_graph if n.id == int(iv.target)), None)
        if node:
            payload["node_title"] = node.title
            payload["node_description"] = node.description

    prompt = f"""
You are the diegetic narrator of a mystery game's drama manager.

Render the following intervention as 1 to 3 sentences of in-world prose. Do NOT say "hint",
"system", "drama manager", or "plot point". Just describe atmosphere, sensation, or a small
narrated event. Keep the tone consistent with a noir detective story.

Intervention: {json.dumps(payload, indent=2)}

Player is at: {state.player_location}.
Recently discovered clue ids: {state.discovered_clues[-3:]}.

Return only the prose. No JSON, no labels.
""".strip()
    return ask_llm(prompt).strip()


def render_turn(
    world: StoryWorld,
    state: WorldState,
    action: Action,
    step_result: Dict[str, Any],
    iv_text: str,
) -> str:
    parts: List[str] = []
    if not step_result["ok"]:
        parts.append(step_result["message"])
    else:
        gw = world.game_world
        cls = step_result.get("classification", "")
        if action.verb == "move":
            here = gw.get_location(state.player_location)
            parts.append(f"You move to {state.player_location}. {here.description if here else ''}")
        elif action.verb == "examine":
            it = next((i for i in gw.items if i.name == action.target), None)
            if it:
                parts.append(f"You examine the {it.name}. {it.description}")
        elif action.verb == "interview":
            parts.append(f"You take {action.target} aside and ask a few questions.")
        elif action.verb == "ask_about":
            parts.append(f"You press {action.target} about {action.indirect}.")
        elif action.verb == "look":
            here = gw.get_location(state.player_location)
            items_here = [it.name for it in gw.items if it.location == state.player_location]
            npcs_here = [n for n, l in gw.npc_locations.items() if l == state.player_location]
            if here:
                parts.append(here.description)
            if items_here:
                parts.append(f"You see: {', '.join(items_here)}.")
            if npcs_here:
                parts.append(f"Present: {', '.join(npcs_here)}.")
            if here and here.neighbors:
                parts.append(f"Exits: {', '.join(here.neighbors)}.")
        elif action.verb == "accuse":
            if cls == "case_solved":
                parts.append(f"You accuse {action.target}. The pieces fit. The case is closed.")
            else:
                parts.append(f"You accuse {action.target}, but the evidence doesn't hold up. They walk free.")

        for nid in step_result["fired_nodes"]:
            node = next((n for n in world.plot_graph if n.id == nid), None)
            if node:
                parts.append(f"({node.title}.)")

    if iv_text:
        parts.append(iv_text)
    return "\n\n".join(p for p in parts if p)


## Step 8 — Game Loop

Interactive REPL. Each turn:

1. Read free-text input.
2. Parse → structured action (LLM).
3. Engine validates and steps the world.
4. DM decides on an intervention (sampled rollouts).
5. Render the turn's narration.

Type `quit` to exit early; the game also ends on a correct accusation or after `max_turns`.

In [ ]:
def play(world: StoryWorld, max_turns: int = 30) -> WorldState:
    engine = GameEngine(world)
    state = initialize_state(world)
    here = world.game_world.get_location(state.player_location)
    print(f"\n=== {world.crime_details.crime_type.upper()}: {world.crime_details.victim} ===")
    if here:
        print(here.description)
    print(f"\nDeadline: {world.countdown_deadline}")
    print(f"If you fail: {world.countdown_consequence}\n")
    print("Type things like: examine <item>, move to <room>, interview <name>, accuse <name>, look, quit\n")

    while state.turn < max_turns and not state.case_solved:
        try:
            nl = input(">> ").strip()
        except EOFError:
            break
        if not nl:
            continue
        if nl.lower() in ("quit", "exit"):
            print("You walk away from the case.")
            break

        action = parse_action(world, state, nl)
        result = engine.step(state, action)
        iv = drama_manager_decide(world, engine, state)
        apply_intervention(world, state, iv)
        iv_text = render_intervention(world, state, iv) if iv.kind != "noop" else ""
        print(render_turn(world, state, action, result, iv_text))
        print()

        if state.case_solved:
            print("*** Case closed. ***")
            break
        if result.get("classification") == "false_accusation":
            print("*** Wrong call. The case slips away. ***")
            break

    return state


# Uncomment to play interactively (requires a working LLM API key):
play(world)



=== MURDER: Alistair Finch, 62, a renowned, cutthroat art collector and philanthropist ===
The grand and opulent main entrance hall of the Azure Bloom Hotel, now a hub of hushed activity and concerned guests being ushered away by staff. Marble floors gleam under crystal chandeliers.

Deadline: By 2:30 AM, when the Azure Bloom Hotel staff complete their initial 'discreet incident cleanup' protocol for Alistair Finch's private lounge.
If you fail: During this rushed cleanup, Eleanor Vance, leveraging her intimate knowledge of Finch's habits and the hotel's operational procedures, will have covertly retrieved and permanently disposed of the custom vape pen. Without this crucial piece of physical evidence—the direct murder weapon containing the lethal dose of respiratory depressant—Finch's death will be officially declared a sudden, natural medical event, making it impossible to prove he was murdered or to ever link Eleanor Vance to the crime.

Type things like: examine <item>, move to <r

WorldState(player_location='Hotel Management Office', inventory=['Poisoned_Vape_Pen', 'Eleanor_Decoy_Vape_Pen'], discovered_clues=[1, 5, 4, 2, 7, 8, 6], interviewed=['Eleanor Vance', 'Cassandra "Cassie" Bell', 'Serena Finch'], eliminated_suspects=[], flags={'Eleanor_Vance_Observed_Suspiciously': True, 'Eleanor_Vance_Detained': True, 'Matching_Micro_Scratches_Confirmed': True, 'Non_Commercial_Tranquilizer_Found': True}, executed_nodes=[1, 2, 3, 4, 5, 6, 8], perm_denied_nodes=[], temp_denied_nodes={}, turn=10, intervention_count=7, case_solved=True, last_classification='case_solved')

## Step 9 — Evaluation Harness

Non-interactive auto-play with the metrics from the project plan:

- **turns_used** — proxy for story length
- **clues_discovered** — does the player actually uncover the mystery?
- **wasted_actions** — actions that neither advance a node nor reveal a clue
- **interventions** — manipulation budget consumed by the DM
- **executed_nodes** — fraction of the plot graph the player walked through

The auto-players (`scripted_player` and `random_player`) bypass the LLM action parser so you can run quick sanity checks without burning API quota.

In [ ]:
def scripted_player(world: StoryWorld, state: WorldState) -> Action:
    return player_model_action(world, state)


def random_player(world: StoryWorld, state: WorldState) -> Action:
    gw = world.game_world
    options: List[Action] = []
    for it in gw.items:
        if it.location == state.player_location:
            options.append(Action(verb="examine", target=it.name))
    for name, loc in gw.npc_locations.items():
        if loc == state.player_location and name not in state.interviewed:
            options.append(Action(verb="interview", target=name))
    here = gw.get_location(state.player_location)
    if here:
        for n in here.neighbors:
            options.append(Action(verb="move", target=n))
    if not options:
        return Action(verb="look")
    return random.choice(options)


def run_eval(
    world: StoryWorld,
    player_fn: Callable[[StoryWorld, WorldState], Action],
    with_dm: bool = True,
    max_turns: int = 30,
) -> Dict[str, Any]:
    engine = GameEngine(world)
    state = initialize_state(world)
    waste = 0
    for _ in range(max_turns):
        if state.case_solved:
            break
        action = player_fn(world, state)
        prev_clues = len(state.discovered_clues)
        result = engine.step(state, action)
        if result["ok"] and not result["fired_nodes"] and len(state.discovered_clues) == prev_clues:
            if action.verb not in ("accuse",):
                waste += 1
        if with_dm:
            iv = drama_manager_decide(world, engine, state, num_samples=2, depth=3)
            apply_intervention(world, state, iv)
    return {
        "turns_used": state.turn,
        "case_solved": state.case_solved,
        "clues_discovered": len(state.discovered_clues),
        "executed_nodes": len(state.executed_nodes),
        "total_nodes": len(world.plot_graph),
        "interventions": state.intervention_count,
        "wasted_actions": waste,
    }


# Quick comparison run
random.seed(0)
print("Scripted player + DM   :", run_eval(world, scripted_player, with_dm=True,  max_turns=25))
random.seed(0)
print("Scripted player no DM  :", run_eval(world, scripted_player, with_dm=False, max_turns=25))
random.seed(0)
print("Random player + DM     :", run_eval(world, random_player,   with_dm=True,  max_turns=25))
random.seed(0)
print("Random player no DM    :", run_eval(world, random_player,   with_dm=False, max_turns=25))


Scripted player + DM   : {'turns_used': 6, 'case_solved': True, 'clues_discovered': 5, 'executed_nodes': 4, 'total_nodes': 15, 'interventions': 4, 'wasted_actions': 5}
Scripted player no DM  : {'turns_used': 15, 'case_solved': True, 'clues_discovered': 5, 'executed_nodes': 0, 'total_nodes': 15, 'interventions': 0, 'wasted_actions': 9}
Random player + DM     : {'turns_used': 10, 'case_solved': True, 'clues_discovered': 7, 'executed_nodes': 15, 'total_nodes': 15, 'interventions': 6, 'wasted_actions': 8}
Random player no DM    : {'turns_used': 25, 'case_solved': False, 'clues_discovered': 5, 'executed_nodes': 0, 'total_nodes': 15, 'interventions': 0, 'wasted_actions': 20}
